# word2vec model
Word2vec is one of the most popular and widely used models for generating
the word embeddings. The embedding generated by the word2vec model captures the syntactic and semantic meanings of a word.

# Building the word2vec model using gensim
Gensim is one of the popular scientific software packages widely used for building vector space models.

In [1]:
!pip install numpy==1.23.5 scipy==1.10.1 gensim==4.3.1

Import the necessary libraries:

In [2]:
import warnings
warnings.filterwarnings(action='ignore')
#data processing
import pandas as pd
import re
from nltk.corpus import stopwords
stopWords = stopwords.words('english')
#modelling
from gensim.models import Word2Vec
from gensim.models import Phrases
from gensim.models.phrases import Phraser

In [11]:
data = pd.read_csv('/content/text.csv',header=None)

In [12]:
data.head()

,0
0,room kind clean strong smell dogs. generally a...
1,stayed crown plaza april april . staff friendl...
2,booked hotel hotwire lowest price could find. ...
3,stayed husband sons way alaska cruise. loved h...
4,girlfriends stayed celebrate th birthdays. pla...


## Preprocessing and preparing the dataset

## A function for preprocessing the dataset

In [13]:
def pre_process(text):
  # convert to lowercase
  text = str(text).lower()
  # remove all special characters and keep only alpha numeric characters and spaces
  text = re.sub(r'[^A-Za-z0-9\s.]',r'',text)
  #remove new lines
  text = re.sub(r'\n',r' ',text)
  # remove stop words
  text = " ".join([word for word in text.split() if word not in stopWords])
  return text

In [14]:
pre_process(data[0][50])

'agree fancy. everything needed. breakfast pool hot tub nice shuttle airport later checkout time. noise issue tough sleep through. awhile forget noisy door nearby noisy guests. complained management later email credit compd us amount requested would return.'

Preprocess the whole dataset:

In [15]:
data[0] = data[0].map(lambda x: pre_process(x))

Each row in our data contains a set of sentences. So, we split
them by '.' and convert them into a list:

In [16]:
data[0][1].split('.')[:5]

['stayed crown plaza april april ',
 ' staff friendly attentive',
 ' elevators tiny ',
 ' food restaurant delicious priced little high side',
 ' course washington dc']

We have the data in a list. But we need to convert them
into a list of lists. So, now again we split it by a space ' '. That is, first, we
split the data by '.' and then we split them by ' ' so that we can get our data
in a list of lists:

In [17]:
corpus = []
for line in data[0][1].split('.'):
  words = [x for x in line.split()]
  corpus.append(words)

In [18]:
corpus[:2]

[['stayed', 'crown', 'plaza', 'april', 'april'],
 ['staff', 'friendly', 'attentive']]

Convert the whole text in our dataset to a list of lists:

In [19]:
data = data[0].map(lambda x: x.split('.'))
corpus = []
for i in (range(len(data))):
  for line in data[i]:
    words = [x for x in line.split()]
    corpus.append(words)

print(corpus[:2])

[['room', 'kind', 'clean', 'strong', 'smell', 'dogs'], ['generally', 'average', 'ok', 'overnight', 'stay', 'youre', 'fussy']]


Now, the problem we have is that our corpus contains only unigrams and it
will not give us results when we give a bigram as an input, for example, san
francisco.
So we use gensim's Phrases functions, which collects all the words that occur
together and adds an underscore between them. So, now san francisco
becomes san_francisco.
We set the min_count parameter to 25, which implies that we ignore all the
words and bigrams that appear less the min_count:

In [20]:
phrases = Phrases(sentences=corpus,min_count=25,threshold=50)
bigram = Phraser(phrases)
for index,sentence in enumerate(corpus):
  corpus[index] = bigram[sentence]

In [21]:
corpus[111]

['connected', 'rivercenter', 'mall', 'downtown', 'san_antonio']

# Building the model

Let's define some of the important
hyperparameters that our model needs:
- The size parameter represents the size of the vector, that is, dimensions
of our vector, to represent a word. The size can be chosen according to
our data size. If our data is very small, then we can set the size to a
small value, but if we have a significantly large dataset, then we can set
the size to 300. In our case, we set the size to 100.
- The window_size parameter represents the distance that should be
considered between the target word and its neighboring word. Words
exceeding the window size from the target word will not be considered
for learning. Typically, a small window size is preferred.
- The min_count parameter represents the minimum frequency of words. If
the particular word's occurrence is less than a min_count, then we can
simply ignore that word.
- The workers parameter specifies the number of worker threads we need
to train the model.
- Setting sg=1 implies that we use the skip-gram model for training, but if it
is set to sg=0, then it implies that we use CBOW model for training.

Define all the hyperparameters using following code:

In [22]:
size = 100
window_size = 2
epochs = 100
min_count = 2
workers = 4
sg = 1

Let's train the model using the Word2Vec function from gensim:

In [28]:
model = Word2Vec(corpus, sg=1,window=window_size,min_count=min_count,workers=workers,epochs=epochs, batch_words=size)

In [31]:
model.save('/content/word2vec.model')

# Evaluating the Embeddings

In [33]:
model = Word2Vec.load('/content/word2vec.model')

We can see in the following code, given san_diego as an input, we are
getting all the other related city names that are most similar:

In [41]:
# wv => the word vector lookup
model.wv.most_similar('san_diego')

[('san_antonio', 0.803612470626831),
 ('memphis', 0.7755193710327148),
 ('austin', 0.7526881098747253),
 ('boston', 0.7519133687019348),
 ('indianapolis', 0.7396535873413086),
 ('seattle', 0.7308306097984314),
 ('dallas', 0.72647625207901),
 ('san_francisco', 0.7226020693778992),
 ('phoenix', 0.7177706360816956),
 ('sd', 0.7111939787864685)]

In [38]:
model.wv.most_similar(positive=['woman', 'king'], negative=['man'], topn=1)

[('queen', 0.7042723298072815)]

In [40]:
text = ['los_angeles','indianapolis', 'holiday', 'san_antonio','new_york']
model.wv.doesnt_match(text)

'holiday'

# Visualizing word embeddings in TensorBoard

In [20]:
!pip uninstall numpy --quiet

!pip install numpy --quiet


Proceed (Y/n)? Y
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 109.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scipy 1.10.1 requires numpy<1.27.0,>=1.19.5, but you have numpy 2.2.6 which is incompatible.
jax 0.6.1 requires scipy>=1.11.1, but you have scipy 1.10.1 which is incompatible.
jaxlib 0.6.1 requires scipy>=1.11.1, but you have scipy 1.10.1 which is incompatible.
tensorflow 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.1 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
cvxpy 1.6.5 requires scipy>=1.11.0, but you have scipy 1.10.1 which is incompatible.
scikit-image

In [ ]:
from tensorboard.plugins import projector
import gensim
import os
import numpy as np
import tensorflow as tf
import os

In [6]:
file_name = "/content/word2vec.model"
model = gensim.models.keyedvectors.KeyedVectors.load(file_name)

After loading the model, we will save the number of words in our model to
the max_size variable:

In [8]:
max_size = len(model.wv.index_to_key) - 1

So, we
initialize a matrix named w2v with the shape as our max_size, which is the
vocabulary size, and the model's first layer size, which is the number of
neurons in the hidden layer:

In [11]:
w2v = np.zeros((max_size,model.layer1_size))

Now, we create a new file called metadata.tsv, where we save all the words in
our model and we store the embedding of each word in the w2v matrix:

In [15]:
if not os.path.exists('projections'):
  os.makedirs('projections')
  with open("projections/metadata.tsv", 'w+') as file_metadata:
    for i, word in enumerate(model.wv.index_to_key[:max_size]):
      #store the embeddings of the word
      w2v[i] = model.wv[word]
      #write the word to a file
      file_metadata.write(word + '\n')

We initialize the TensorFlow session:

In [16]:
sess = tf.InteractiveSession()

NameError: name 'tf' is not defined

Initialize the TensorFlow variable called embedding that holds the word
embeddings:

In [ ]:
with tf.device("/cpu:0"):
  embedding = tf.Variable(w2v, trainable=False, name='embedding')

Initialize all the variables:

In [ ]:
tf.global_variables_initializer().run()

Create an object to the saver class, which is actually used for saving and
restoring variables to and from our checkpoints:

In [ ]:
saver = tf.train.Saver()

Using FileWriter, we can save our summaries and events to our event file:

In [ ]:
writer = tf.summary.FileWriter('projections', sess.graph)

Now, we initialize the projectors and add the embeddings:

In [ ]:
config = projector.ProjectorConfig()
embed = config.embeddings.add()

Next, we specify our tensor_name as embedding and metadata_path to the metadata.tsv
file, where we have the words:

In [ ]:
embed.tensor_name = 'embedding'
embed.metadata_path = 'metadata.tsv'

And, finally, save the model:

In [ ]:
projector.visualize_embeddings(writer, config)
saver.save(sess, 'projections/model.ckpt', global_step=max_size)

Now, open the terminal and type the following command to open the
tensorboard:

In [ ]:
!tensorboard --logdir=projections --port=8000

Once the TensorBoard is opened, go to the PROJECTOR tab. We can see the
output, as shown in the following screenshot. As you can notice, when we
type the word delighted, we can see all the related words, such as pleasant,
surprise, and many more similar words, adjacent to that: